# AR Step 3 MLP Probes

Compares ridge probes and MLP probes on the same AR latent space and the same raw/residual targets from Step 3.


In [ ]:
from pathlib import Path
import json
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

def find_project_root(start=None):
    p = (Path(start) if start is not None else Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / ".git").exists():
            return parent
    raise RuntimeError(f"Couldn't find project root from {p}")

PROJECT_ROOT = find_project_root(start=".ser/projects/Smiles-latent-project")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from artifacts.model_compare.ar_model_h256_l256.patched_notebooks import ar_common as arc

ARTIFACT_ROOT = arc.ARTIFACT_ROOT
print("artifact_root =", ARTIFACT_ROOT)
for warning_text in arc.dependency_report():
    warnings.warn(warning_text)


In [ ]:
OUTPUT_DIR = ARTIFACT_ROOT / "confounds_mlp_step3"
TABLES_DIR = OUTPUT_DIR / "tables"
PLOTS_DIR = OUTPUT_DIR / "plots"
for path in [OUTPUT_DIR, TABLES_DIR, PLOTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

STEP3_DIR = ARTIFACT_ROOT / "confounds_step3"
STEP3_TABLES = STEP3_DIR / "tables"


## Load Existing Step 3 Artifacts


In [ ]:
panel_path = STEP3_DIR / "panels_step3_with_residuals.csv"
z_path = STEP3_DIR / "Z_mu.npy"
if not panel_path.exists() or not z_path.exists():
    raise FileNotFoundError("Run ar_step3_confounds.ipynb before the MLP probe notebook.")

panel = pd.read_csv(panel_path)
Z = np.load(z_path).astype(np.float32)
split = panel["split"].astype(str).to_numpy()
ridge_raw = pd.read_csv(STEP3_TABLES / "r2_Z_to_Y.csv")
ridge_resid = pd.read_csv(STEP3_TABLES / "r2_Z_to_Y_vs_Yresid.csv")
corr_path = STEP3_TABLES / "property_confound_correlations.csv"
corr_long = pd.read_csv(corr_path) if corr_path.exists() else pd.DataFrame()

property_cols = [col for col in arc.AR_PROPERTY_COLUMNS if col in panel.columns and panel[col].notna().sum() > 100]
print("properties =", property_cols)
print("latent shape =", Z.shape)


## MLP Helper Functions


In [ ]:
import copy
torch = arc.torch
nn = torch.nn
DataLoader = torch.utils.data.DataLoader
TensorDataset = torch.utils.data.TensorDataset

BATCH_SIZE = 8192
MAX_EPOCHS = 15
PATIENCE = 4
LR = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN_WIDTH = 256
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class MLPProbe(nn.Module):
    def __init__(self, in_dim, hidden_width=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_width),
            nn.GELU(),
            nn.LayerNorm(hidden_width),
            nn.Linear(hidden_width, hidden_width),
            nn.GELU(),
            nn.Linear(hidden_width, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

def fit_mlp_probe(X, y, split):
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    finite = np.isfinite(y) & np.isfinite(X).all(axis=1)
    masks = {name: (split == name) & finite for name in ["train", "val", "test"]}

    x_scaler = arc.StandardScaler().fit(X[masks["train"]])
    y_mean = float(y[masks["train"]].mean())
    y_std = float(y[masks["train"]].std() + 1e-8)
    Xs = x_scaler.transform(X).astype(np.float32)
    ys = ((y - y_mean) / y_std).astype(np.float32)

    def loader_for(mask, shuffle=False):
        ds = TensorDataset(torch.from_numpy(Xs[mask]), torch.from_numpy(ys[mask]))
        return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

    model = MLPProbe(X.shape[1], HIDDEN_WIDTH).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.MSELoss()
    best_state = None
    best_val = np.inf
    stale = 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        for xb, yb in loader_for(masks["train"], shuffle=True):
            xb = xb.to(device)
            yb = yb.to(device)
            loss = loss_fn(model(xb), yb)
            opt.zero_grad()
            loss.backward()
            opt.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in loader_for(masks["val"], shuffle=False):
                xb = xb.to(device)
                yb = yb.to(device)
                val_losses.append(float(loss_fn(model(xb), yb).detach().cpu()))
        val_loss = float(np.mean(val_losses))
        if val_loss < best_val - 1e-5:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
        if stale >= PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    preds = {}
    model.eval()
    with torch.no_grad():
        for name in ["train", "val", "test"]:
            out = []
            for xb, _ in loader_for(masks[name], shuffle=False):
                pred = model(xb.to(device)).detach().cpu().numpy()
                out.append(pred)
            preds[name] = np.concatenate(out) * y_std + y_mean

    return {
        "r2_train": arc.safe_r2(y[masks["train"]], preds["train"]),
        "r2_val": arc.safe_r2(y[masks["val"]], preds["val"]),
        "r2_test": arc.safe_r2(y[masks["test"]], preds["test"]),
        "epochs": int(epoch + 1),
    }


## Fit Ridge And MLP On Raw And Residual Targets


In [ ]:
raw_rows = []
resid_rows = []

for prop in tqdm(property_cols, desc="raw MLP probes"):
    y = panel[prop].to_numpy(dtype=float)
    mlp_scores = fit_mlp_probe(Z, y, split)
    raw_rows.append({"property": prop, **mlp_scores})

for prop in tqdm(property_cols, desc="residual MLP probes"):
    resid_col = f"resid_{prop}"
    if resid_col not in panel.columns:
        continue
    y = panel[resid_col].to_numpy(dtype=float)
    mlp_scores = fit_mlp_probe(Z, y, split)
    resid_rows.append({"property": prop, "target": resid_col, **mlp_scores})

mlp_raw = pd.DataFrame(raw_rows)
mlp_resid = pd.DataFrame(resid_rows)
mlp_raw.to_csv(TABLES_DIR / "r2_Z_to_Y_mlp.csv", index=False)
mlp_resid.to_csv(TABLES_DIR / "r2_Z_to_Yresid_mlp.csv", index=False)
display(mlp_raw)
display(mlp_resid)


## Compare MLP To Ridge


In [ ]:
raw_compare = mlp_raw.merge(
    ridge_raw[["property", "r2_test"]].rename(columns={"r2_test": "ridge_raw_r2_test"}),
    on="property",
    how="left",
)
raw_compare["mlp_raw_r2_test"] = raw_compare["r2_test"]
raw_compare["Delta-R2_raw"] = raw_compare["mlp_raw_r2_test"] - raw_compare["ridge_raw_r2_test"]
raw_compare.to_csv(TABLES_DIR / "mlp_vs_ridge_raw.csv", index=False)

resid_compare = mlp_resid.merge(
    ridge_resid[["property", "r2_residual_test"]].rename(columns={"r2_residual_test": "ridge_residual_r2_test"}),
    on="property",
    how="left",
)
resid_compare["mlp_residual_r2_test"] = resid_compare["r2_test"]
resid_compare["Delta-R2_residual"] = resid_compare["mlp_residual_r2_test"] - resid_compare["ridge_residual_r2_test"]
resid_compare.to_csv(TABLES_DIR / "mlp_vs_ridge_resid.csv", index=False)

overview = raw_compare[["property", "ridge_raw_r2_test", "mlp_raw_r2_test", "Delta-R2_raw"]].merge(
    resid_compare[["property", "ridge_residual_r2_test", "mlp_residual_r2_test", "Delta-R2_residual"]],
    on="property",
    how="outer",
)

if len(corr_long):
    max_corr = corr_long.assign(abs_spearman=lambda d: d["spearman_corr"].abs()).groupby("property", as_index=False)["abs_spearman"].max()
    overview = overview.merge(max_corr, on="property", how="left")
else:
    overview["abs_spearman"] = np.nan

def classify(row):
    if row.get("abs_spearman", 0) >= 0.80:
        return "confounded property"
    if row.get("ridge_residual_r2_test", 0) >= 0.50 and row.get("Delta-R2_residual", 0) < 0.10:
        return "globally linear property"
    if row.get("Delta-R2_residual", 0) >= 0.15:
        return "nonlinear/local property"
    if row.get("mlp_residual_r2_test", 0) < 0.20:
        return "weakly represented property"
    return "mixed property"

overview["interpretation"] = overview.apply(classify, axis=1)
overview.to_csv(TABLES_DIR / "mlp_ridge_delta_r2_overview.csv", index=False)
display(overview.sort_values("Delta-R2_residual", ascending=False))


## Minimal Plots And Summary


In [ ]:
def barh_metric(df, value, title, filename):
    plot_df = df.sort_values(value, ascending=True)
    plt.figure(figsize=(8, max(4, 0.35 * len(plot_df))))
    plt.barh(plot_df["property"], plot_df[value])
    plt.axvline(0, color="black", linewidth=1)
    plt.xlabel(value)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / filename, dpi=180)
    plt.show()

barh_metric(raw_compare, "Delta-R2_raw", "Delta-R2 raw: MLP minus ridge", "delta_r2_raw.png")
barh_metric(resid_compare, "Delta-R2_residual", "Delta-R2 residual: MLP minus ridge", "delta_r2_residual.png")

plt.figure(figsize=(5, 5))
plt.scatter(raw_compare["ridge_raw_r2_test"], raw_compare["mlp_raw_r2_test"])
lims = [0, 1]
plt.plot(lims, lims, color="black", linewidth=1)
plt.xlim(lims)
plt.ylim(lims)
plt.xlabel("ridge raw R^2")
plt.ylabel("MLP raw R^2")
plt.title("Ridge vs MLP raw R^2")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "ridge_vs_mlp_raw_r2.png", dpi=180)
plt.show()

plt.figure(figsize=(5, 5))
plt.scatter(resid_compare["ridge_residual_r2_test"], resid_compare["mlp_residual_r2_test"])
plt.plot(lims, lims, color="black", linewidth=1)
plt.xlim(lims)
plt.ylim(lims)
plt.xlabel("ridge residual R^2")
plt.ylabel("MLP residual R^2")
plt.title("Ridge vs MLP residual R^2")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "ridge_vs_mlp_residual_r2.png", dpi=180)
plt.show()

summary = {
    "artifact_root": str(OUTPUT_DIR),
    "n_properties": int(len(property_cols)),
    "avg_raw_Delta-R2": float(raw_compare["Delta-R2_raw"].mean()),
    "avg_residual_Delta-R2": float(resid_compare["Delta-R2_residual"].mean()),
    "properties": property_cols,
}
arc.write_json(OUTPUT_DIR / "summary_mlp_confounds.json", summary)
display(pd.DataFrame([summary]))
